In [1]:
import sys
sys.path.append('/home/pavel/pyadi-iio/')
sys.path.append('/home/pavel/libiio')
import adi
import matplotlib.pyplot as plt
import numpy as np

/home/pavel/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [3]:
# Create radio
sdr = adi.ad9363("ip:192.168.2.1")

sample_rate = 20e6 # Hz
center_freq = 2.71e9 # Hz
num_samps = 8192 # number of samples per call to rx()

sdr.sample_rate = int(sample_rate)
sdr.tx_enabled_channels = [0, 1]
sdr.tx_rf_bandwidth = int(sample_rate) # filter cutoff, just set it to the same as sample rate
sdr.tx_lo = int(center_freq)

sdr.gain_control_mode_chan0 = "manual"
sdr.gain_control_mode_chan1 = "manual"
sdr.tx_hardwaregain_chan0 = int(-10)
sdr.tx_hardwaregain_chan1 = int(-10)
sdr.tx_cyclic_buffer = True

In [4]:
def golay_complementary_pair(n_bits=4):
    """
    Комплементарная пара Голея (двоичные последовательности с суммарной АКФ как у дельта-импульса).
    Рекурсия: A' = [A, B], B' = [A, -B], старт A=[1,1], B=[1,-1].
    Длина последовательностей = 2**n_bits.
    """
    A = np.array([1, 1], dtype=np.int8)
    B = np.array([1, -1], dtype=np.int8)
    for _ in range(n_bits - 1):
        A, B = np.concatenate([A, B]), np.concatenate([A, -B])
    return A, B


def preamble_with_gaps(
    sample_rate_hz,
    n_bits=4,
    samples_per_chip=8,
    gap_seconds=50e-6,
    num_bursts_in_buffer=4,
):
    """
    Одна «преамбула» = чипы последовательности A (BPSK ±1), затем тишина (нули в базовой полосе).
    Буфер = несколько таких циклов подряд; при cyclic=True Pluto повторяет весь буфер.
    """
    A, _ = golay_complementary_pair(n_bits)
    chips = A.astype(np.float32)
    preamble = np.repeat(chips, samples_per_chip)
    gap_samples = max(0, int(round(gap_seconds * sample_rate_hz)))
    gap = np.zeros(gap_samples, dtype=np.complex64)
    burst = np.concatenate([preamble.astype(np.complex64), gap])
    full = np.tile(burst, num_bursts_in_buffer)
    return full, len(preamble), gap_samples


def preamble_template(n_bits, samples_per_chip):
    """Эталон преамбулы (без паузы), нормированный для согласованного фильтра."""
    A, _ = golay_complementary_pair(n_bits)
    chips = A.astype(np.float64)
    ref = np.repeat(chips, samples_per_chip)
    ref = ref / (np.linalg.norm(ref) + 1e-12)
    return ref.astype(np.complex128)


def matched_filter_preamble(rx, ref):
    """Согласованный фильтр; пик |mf| когда окно совпало с преамбулой."""
    rx = np.asarray(rx).ravel()
    ref = np.asarray(ref).ravel()
    return np.convolve(rx, np.conj(ref[::-1]), mode="full")


def detect_preamble_starts(mf, ref_len, min_height_ratio=0.25):
    """Грубый поиск начал преамбул по локальным максимумам |mf|."""
    mag = np.abs(mf)
    thr = np.max(mag) * min_height_ratio
    peaks = []
    for i in range(1, len(mag) - 1):
        if mag[i] >= mag[i - 1] and mag[i] >= mag[i + 1] and mag[i] >= thr:
            peaks.append(i)
    if not peaks:
        return np.array([], dtype=int), mag
    peaks = np.array(peaks)
    starts = peaks - (ref_len - 1)
    return starts, mag

In [5]:
# --- параметры ---
n_bits = 4
samples_per_chip = 8
gap_seconds = 100e-6  # пауза между преамбулами (только нули)
num_bursts = 4

samples_1d, n_preamble_samp, n_gap_samp = preamble_with_gaps(
    sample_rate,
    n_bits=n_bits,
    samples_per_chip=samples_per_chip,
    gap_seconds=gap_seconds,
    num_bursts_in_buffer=num_bursts,
)

samples_1d = samples_1d * (0.5 * 2**14)  # амплитуда как в вашем примере с синусом

signal = np.array([samples_1d, samples_1d])
print(
    f"Преамбула: {2**n_bits} чипов × {samples_per_chip} отсч/чип = {n_preamble_samp} отсч; "
    f"пауза: {n_gap_samp} отсч (~{n_gap_samp/sample_rate*1e6:.1f} µs); "
    f"период буфера: {signal.shape[1]} отсч"
)

Преамбула: 16 чипов × 8 отсч/чип = 128 отсч; пауза: 2000 отсч (~100.0 µs); период буфера: 8512 отсч


In [10]:
sdr.tx(signal)

In [9]:
# Stop transmitting
sdr.tx_destroy_buffer()

## Приёмник

Та же Pluto, `sample_rate` и `center_freq`, что выше. Перед приёмом можно оставить передачу (`sdr.tx(signal)`) или выключить TX — при кабельном/ближнем приёме настройте `rx_hardwaregain`, чтобы не было насыщения.

In [ ]:
# Эталон и буфер приёма (использует n_preamble_samp, n_gap_samp, num_bursts, n_bits из ячейки параметров)
burst_period = n_preamble_samp + n_gap_samp
ref = preamble_template(n_bits, samples_per_chip)
ref_len = len(ref)

sdr.rx_enabled_channels = [0, 1]
sdr.rx_rf_bandwidth = int(sample_rate)
sdr.rx_lo = int(center_freq)
sdr.rx_buffer_size = max(16384, burst_period * num_bursts + ref_len)
sdr.gain_control_mode_chan0 = "manual"
sdr.gain_control_mode_chan1 = "manual"
sdr.rx_hardwaregain_chan0 = int(30)
sdr.rx_hardwaregain_chan1 = int(30)
print(f"Эталон {ref_len} отсч, rx_buffer_size={sdr.rx_buffer_size}")

In [ ]:
rx = sdr.rx()
rx0 = rx[0]

mf = matched_filter_preamble(rx0, ref)
starts, _ = detect_preamble_starts(mf, ref_len)

peak_idx = int(np.argmax(np.abs(mf)))
est_start = peak_idx - (ref_len - 1)
print(f"Пик MF: индекс {peak_idx}, оценка начала преамбулы в rx: {est_start}")
if len(starts):
    print(f"Пиков (грубо): {len(starts)}, старты: {starts[:8]}")

fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=False)
n_plot = min(len(rx0), 3 * burst_period)
ax[0].plot(np.real(rx0[:n_plot]), label="Re", alpha=0.9)
ax[0].plot(np.imag(rx0[:n_plot]), label="Im", alpha=0.7)
ax[0].set_title("Принятый сигнал (фрагмент)")
ax[0].set_xlabel("отсчёт")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

lag = np.arange(len(mf)) - (ref_len - 1)
ax[1].plot(lag, np.abs(mf))
ax[1].axvline(est_start, color="r", linestyle="--", alpha=0.7, label="оценка старта")
ax[1].set_title("|Согласованный фильтр| (лаг → начало преамбулы в rx)")
ax[1].set_xlabel("лаг (отсчёты)")
ax[1].legend()
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()